# 06 — Natural Language Processing (NLP)

This notebook demonstrates basic text normalization, tokenization, bag-of-words, TF-IDF, and text classification. The small dataset is synthetic and generated locally.

## 1. Setup

Install the dependencies if needed: `python -m pip install numpy pandas scikit-learn matplotlib jupyter`.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

SEED = 42
np.random.seed(SEED)
print('NumPy:', np.__version__)

## 2. Text normalization

Preprocessing is task-dependent. This simple example lowercases text, removes URLs, and normalizes whitespace; it deliberately keeps punctuation and negation words.

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

samples = [
    '  This product is GREAT!  ',
    'Read https://example.com/page for details',
    'I do not recommend this at all.',
]
for sample in samples:
    print(repr(normalize_text(sample)))

## 3. Tokenization and bag-of-words

A basic split is useful for demonstration but is not a complete language-aware tokenizer. `CountVectorizer` builds a vocabulary and converts documents to token counts.

In [ ]:
sentence = 'NLP is useful, but NLP is not magic!'
print('Whitespace tokens:', sentence.split())

docs = ['cats like milk', 'dogs like walks', 'cats and dogs']
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(docs)
print('Vocabulary:', count_vectorizer.get_feature_names_out())
print('Count matrix:\n', count_matrix.toarray())

## 4. TF-IDF

TF-IDF downweights terms that appear in many documents and can emphasize terms that help distinguish documents. For real evaluation, fit the vectorizer on training text only.

In [ ]:
tfidf_demo = TfidfVectorizer()
tfidf_matrix = tfidf_demo.fit_transform(docs)
print('TF-IDF vocabulary:', tfidf_demo.get_feature_names_out())
print('TF-IDF matrix:\n', np.round(tfidf_matrix.toarray(), 3))

## 5. Create a small synthetic sentiment dataset

The examples below are intentionally simple and repetitive so you can inspect model behavior. They are not representative of real-world language, and metrics on this toy data should not be interpreted as real-world performance.

In [ ]:
positive_templates = [
    'I love this product', 'This is excellent', 'Really happy with the service',
    'Great quality and fast delivery', 'I recommend this to everyone',
    'The experience was wonderful', 'Very useful and easy to use',
    'Fantastic result, I am pleased', 'The support was helpful',
    'A brilliant and enjoyable experience',
]
negative_templates = [
    'I hate this product', 'This is terrible', 'Really unhappy with the service',
    'Poor quality and slow delivery', 'I do not recommend this',
    'The experience was awful', 'Very confusing and hard to use',
    'Disappointing result, I am upset', 'The support was unhelpful',
    'A frustrating and unpleasant experience',
]
texts, labels = [], []
for _ in range(12):
    texts.extend(positive_templates)
    labels.extend([1] * len(positive_templates))
    texts.extend(negative_templates)
    labels.extend([0] * len(negative_templates))
# Add mild variations to avoid an entirely identical repeated set.
texts = [normalize_text(t) for t in texts]
labels = np.array(labels, dtype=int)
print('Number of examples:', len(texts))
print('Class counts:', {int(k): int(v) for k, v in zip(*np.unique(labels, return_counts=True))})

## 6. Split data and build a leakage-safe pipeline

The split happens before fitting TF-IDF. The pipeline fits text features and the classifier on training data together.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=SEED, stratify=labels
)
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=SEED)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, predictions), 4))
print(classification_report(y_test, predictions, target_names=['negative', 'positive'], zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, display_labels=['negative', 'positive'], ax=ax, cmap=None
)
ax.set_title('Toy sentiment classifier — confusion matrix')
plt.tight_layout()
plt.show()

## 7. Inspect predictions

Looking at errors can reveal issues hidden by aggregate metrics. The synthetic examples may be too easy; use a more diverse and appropriately licensed dataset for a serious project.

In [ ]:
for text, true_label, pred_label in zip(X_test[:12], y_test[:12], predictions[:12]):
    print(f'true={true_label} predicted={pred_label} | {text}')

errors = [(t, int(y), int(p)) for t, y, p in zip(X_test, y_test, predictions) if y != p]
print('\nNumber of errors:', len(errors))
for text, true_label, pred_label in errors[:5]:
    print(f'true={true_label} predicted={pred_label} | {text}')

## 8. Optional experiments

Try unigram-only features (`ngram_range=(1, 1)`), a different classifier, or a few additional ambiguous examples. Re-split with a fixed seed for comparability. For repeated or near-duplicate examples, consider a group-based split so copies cannot leak across train and test.

## Key takeaways

- Normalize text according to the task, not by blindly removing tokens.
- Bag-of-words and TF-IDF are useful baselines.
- Fit vectorizers only on training data; pipelines help avoid leakage.
- Evaluate with per-class metrics and inspect errors.
- Real-world NLP requires attention to data quality, privacy, representativeness, and bias.

Next: complete `practice.md`, then review `solution.md`.